In [2]:

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']
newsgroups = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))

tfidf = TfidfVectorizer(max_features=1000, min_df=5, max_df=0.5, stop_words='english')
x_tfidf = tfidf.fit_transform(newsgroups.data)

print(f'원본 TF-IDF 행렬 차원 : {x_tfidf.shape}')

원본 TF-IDF 행렬 차원 : (2034, 1000)


In [4]:
from sklearn.decomposition import TruncatedSVD
# 10개의 주성분(토픽)으로 축소
svd = TruncatedSVD(n_components=10, random_state=42)
x_svd = svd.fit_transform(x_tfidf)
x_svd.shape, svd.explained_variance_ratio_.sum()

((2034, 10), 0.07147410086048102)

In [6]:
# 잠재의미(토픽)별 핵심단어 확인
import numpy as np 
terms = tfidf.get_feature_names_out()
for i, comp in enumerate(svd.components_):
    # 각 컴포넌트에서 가중치가 높은 상위 10개 단어 추출
    top_terms_index = np.argsort(-comp)[:10]
    top_terms = [terms[idx] for idx in top_terms_index]
    print(f'topic {i+1}: {", ".join(top_terms)}') 

topic 1: don, people, just, god, think, space, like, know, does, good
topic 2: god, jesus, people, bible, believe, say, christian, religion, don, think
topic 3: space, nasa, launch, shuttle, moon, orbit, station, lunar, earth, cost
topic 4: god, space, jesus, edu, bible, satan, thanks, believe, does, atheism
topic 5: thanks, does, know, space, anybody, advance, objective, mail, looking, hi
topic 6: objective, morality, moral, values, image, does, data, file, science, problem
topic 7: files, think, space, don, file, god, just, edu, nasa, format
topic 8: edu, graphics, objective, thanks, morality, moral, information, mail, email, university
topic 9: god, graphics, just, like, think, ve, moon, mode, card, comp
topic 10: com, said, bob, away, stay, god, morality, objective, ve, atheism


In [7]:
# 유사도 기반 검색 테스트 (차원축소 효과)
# 단어가 직접 겹치지 않아도 의미적으로 유사한 문서를 찾을수 있는지 확인

from sklearn.metrics.pairwise import cosine_similarity
sim_scores = cosine_similarity(x_svd[0:1], x_svd)
top_index = np.argsort(-sim_scores).reshape(-1)[:5]
print(f'가장유사한 문서 top5 index : {top_index}')
print(f'유사도 점수: {sim_scores[0][top_index]}')

가장유사한 문서 top5 index : [   0 1892  501 1209 1089]
유사도 점수: [1.         0.98926926 0.98200958 0.98009323 0.97241678]
